# Revela — Notebook 03: CNN Cancer-Risk Classification Benchmark Requirements

**Issue:** #108 — P1: Prepare benchmark plan  
**Dataset:** BCN20000  
**Model:** Revela dermoscopic CNN — EfficientNet-B0  

### Purpose
Define benchmark requirements for evaluating the Revela dermoscopic CNN using the new cancer-risk taxonomy. This notebook covers:
- Taxonomy definition (4-class cancer-risk, Option C from D3.1)
- BCN20000 label mapping to the new taxonomy
- Evaluation metrics and safety requirements
- Benchmark dataset selection
- Output template for recording results

### Change from Previous Version
The original approach compared Revela CNN outputs against ChatGPT/Claude LLM text responses. That comparison is not valid — Revela is a CNN classifier that outputs a class label and confidence scores, not a language model. Comparing classification labels against LLM reasoning text is not a meaningful benchmark.

This reworked version defines a proper classification benchmark: CNN predictions evaluated against histopathology-confirmed ground truth labels.

> ⚠️ This is a benchmark requirements document only. Results must not be used to claim clinical superiority over any tool.

## Block 1 — CNN v1 Baseline (3-Class, Reference Only)

In [4]:
baseline_v1 = {
    'Model': 'Revela CNN v1 — EfficientNet-B0',
    'Dataset': 'BCN20000',
    'Classes (old 3-class)': ['Melanoma', 'Benign nevus', 'Other lesion'],
    'Top-1 Accuracy': '76.16%',
    'Macro-F1': '0.7443',
    'Confirm type': 'Histopathology',
    'Image type': 'Dermoscopic',
    'Status': 'Reference baseline only',
}

print('=== REVELA CNN v1 BASELINE (OLD 3-CLASS) ===')
for k, v in baseline_v1.items():
    print(f'  {k:30s}: {v}')

print()
print('PROBLEM: "Other lesion" class mixes malignant (BCC, SCC) and benign lesions.')
print('         Cannot communicate this class as low-risk or non-cancer.')
print('         Model v1 is superseded by cancer-risk taxonomy (D3.1).')

=== REVELA CNN v1 BASELINE (OLD 3-CLASS) ===
  Model                         : Revela CNN v1 — EfficientNet-B0
  Dataset                       : BCN20000
  Classes (old 3-class)         : ['Melanoma', 'Benign nevus', 'Other lesion']
  Top-1 Accuracy                : 76.16%
  Macro-F1                      : 0.7443
  Confirm type                  : Histopathology
  Image type                    : Dermoscopic
  Status                        : Reference baseline only

PROBLEM: "Other lesion" class mixes malignant (BCC, SCC) and benign lesions.
         Cannot communicate this class as low-risk or non-cancer.
         Model v1 is superseded by cancer-risk taxonomy (D3.1).


## Block 2 — Cancer-Risk Taxonomy (4-Class, Option C from D3.1)

In [ ]:
cancer_risk_taxonomy = {
    'Melanoma': {
        'description': 'All melanoma variants including metastatic melanoma',
        'risk_level': 'High cancer risk',
        'bcn20000_labels': ['Melanoma, NOS', 'Melanoma metastasis'],
    },
    'Non-melanoma skin cancer': {
        'description': 'Basal cell carcinoma and squamous cell carcinoma',
        'risk_level': 'High cancer risk',
        'bcn20000_labels': ['Basal cell carcinoma', 'Squamous cell carcinoma, NOS'],
    },
    'Benign nevus': {
        'description': 'Benign melanocytic nevi',
        'risk_level': 'Low risk',
        'bcn20000_labels': ['Nevus'],
    },
    'Other non-cancer / indeterminate lesion': {
        'description': 'Benign lesions and pre-cancer / indeterminate cases (e.g. actinic keratosis)',
        'risk_level': 'Variable — includes pre-cancer',
        'bcn20000_labels': ['Seborrheic keratosis', 'Solar or actinic keratosis',
                             'Solar lentigo', 'Dermatofibroma', 'Scar', 'all other'],
    },
}

print('=== CANCER-RISK 4-CLASS TAXONOMY (Option C — D3.1, finalized #117) ===')
for cls, info in cancer_risk_taxonomy.items():
    print(f'\n  Class        : {cls}')
    print(f'  Risk level   : {info["risk_level"]}')
    print(f'  Description  : {info["description"]}')
    print(f'  BCN20000 src : {info["bcn20000_labels"]}')

## Block 3 — BCN20000 Label Mapping to Cancer-Risk Taxonomy

In [ ]:
import pandas as pd

df = pd.read_csv('../../bcn20000_metadata_2026-05-11.csv')

def map_to_cancer_risk(row):
    d3 = str(row.get('diagnosis_3', '')).lower()
    if 'melanoma' in d3:
        return 'Melanoma'
    if 'basal cell carcinoma' in d3 or 'squamous cell carcinoma' in d3:
        return 'Non-melanoma skin cancer'
    if 'nevus' in d3 or 'nevi' in d3:
        return 'Benign nevus'
    return 'Other non-cancer / indeterminate lesion'

df['cancer_risk_class'] = df.apply(map_to_cancer_risk, axis=1)

print('=== BCN20000 CLASS COUNTS — CANCER-RISK TAXONOMY ===')
total = len(df)
for cls, count in df['cancer_risk_class'].value_counts().items():
    pct = (count / total) * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls:40s}: {count:5,} ({pct:.1f}%) {bar}')

cancer_count = df[df['cancer_risk_class'].isin(
    ['Melanoma', 'Non-melanoma skin cancer']
)].shape[0]
print(f'\n  Total rows              : {total:,}')
print(f'  Cancer cases (combined) : {cancer_count:,} ({cancer_count/total*100:.1f}%)')

# Note: the 4th class here includes rows where diagnosis_3 is missing/unknown (NaN → mapped
# to 'Other non-cancer / indeterminate lesion'). Roman's approved training count (3,121)
# excludes those 1,307 unknown rows. They will be dropped in the #118 splits rebuild.
print('\n=== TOP RAW DIAGNOSIS_3 LABELS ===')
mapping_check = df.groupby(['diagnosis_3', 'cancer_risk_class']).size().reset_index(name='count')
print(mapping_check.sort_values('count', ascending=False).head(15).to_string(index=False))

## Block 4 — Benchmark Dataset Selection

For qualitative review: 6 images per class = 24 images total.  
For full evaluation: use the frozen test split from `data/processed/bcn20000_cancer_risk/test.csv` (produced by #118).

In [ ]:
samples = []
for cls in [
    'Melanoma',
    'Non-melanoma skin cancer',
    'Benign nevus',
    'Other non-cancer / indeterminate lesion'
]:
    subset = df[df['cancer_risk_class'] == cls][
        ['isic_id', 'diagnosis_3', 'cancer_risk_class',
         'age_approx', 'sex', 'anatom_site_general', 'diagnosis_confirm_type']
    ].dropna(subset=['diagnosis_3'])
    samples.append(subset.sample(6, random_state=42))

benchmark_images = pd.concat(samples).reset_index(drop=True)
benchmark_images.index += 1

print(f'Total benchmark images : {len(benchmark_images)}')
print(f'Per class              : 6')
print()
print(benchmark_images.to_string())

benchmark_images.to_csv('../../bcn20000_benchmark_24_images.csv', index=True)
print('\nSaved: bcn20000_benchmark_24_images.csv')

## Block 5 — Evaluation Metrics Requirements

### Safety Metrics — Report First

| Metric | Description | Why |
|--------|-------------|-----|
| **Cancer recall** | Recall for Melanoma + Non-melanoma skin cancer combined | A missed cancer is a false-negative with clinical consequence |
| **Cancer false-negative rate** | 1 − cancer recall | Must be minimised |
| **Melanoma recall** | Recall for Melanoma class specifically | Highest-severity class |
| **NMSC recall** | Recall for Non-melanoma skin cancer specifically | Second cancer class |

### Classification Metrics — Standard

| Metric | Description |
|--------|-------------|
| **Top-1 Accuracy** | Fraction of images where top prediction matches ground truth |
| **Macro-F1** | Unweighted mean F1 across all 4 classes |
| **Balanced Accuracy** | Mean recall across classes — handles class imbalance |
| **Per-class Precision** | TP / (TP + FP) per class |
| **Per-class Recall** | TP / (TP + FN) per class |
| **Per-class F1** | Harmonic mean of precision and recall per class |
| **Confusion Matrix** | Full 4×4 matrix |

### Comparison Baseline
- Compare cancer-risk CNN v2 against old CNN v1 (76.16% top-1, macro-F1 0.7443)
- The v1 model cannot serve as a cancer-risk classifier — comparison is informational only

### Evaluation Split
- Use frozen test split only (`data/processed/bcn20000_cancer_risk/test.csv` from #118)
- Do NOT evaluate on the validation split
- The 24-image sample (Block 4) is for qualitative review only, not metric calculation

## Block 6 — Benchmark Output Template

In [8]:
cols = [
    'isic_id',
    'ground_truth_raw',
    'cancer_risk_class',
    'model_version',
    'predicted_class',
    'top1_correct',
    'confidence_predicted',
    'confidence_melanoma',
    'confidence_nmsc',
    'confidence_benign_nevus',
    'confidence_other',
    'cancer_true_label',
    'cancer_predicted',
    'cancer_correct',
    'notes',
]

template = pd.DataFrame(index=range(len(benchmark_images)), columns=cols)
template['isic_id'] = benchmark_images['isic_id'].values
template['ground_truth_raw'] = benchmark_images['diagnosis_3'].values
template['cancer_risk_class'] = benchmark_images['cancer_risk_class'].values
template['cancer_true_label'] = template['cancer_risk_class'].isin(
    ['Melanoma', 'Non-melanoma skin cancer']
).astype(int)

template.to_csv('../../bcn20000_cancer_risk_benchmark_template.csv', index=False)
print('Benchmark template saved: bcn20000_cancer_risk_benchmark_template.csv')
print(f'Rows   : {len(template)}')
print(f'Columns: {len(cols)}')
print()
print(template[['isic_id', 'ground_truth_raw', 'cancer_risk_class', 'cancer_true_label']].to_string())

Benchmark template saved: bcn20000_cancer_risk_benchmark_template.csv
Rows   : 24
Columns: 15

         isic_id            ground_truth_raw                 cancer_risk_class  cancer_true_label
0   ISIC_0056752               Melanoma, NOS                          Melanoma                  1
1   ISIC_0054874               Melanoma, NOS                          Melanoma                  1
2   ISIC_0063202               Melanoma, NOS                          Melanoma                  1
3   ISIC_0072763               Melanoma, NOS                          Melanoma                  1
4   ISIC_0053576               Melanoma, NOS                          Melanoma                  1
5   ISIC_0071996               Melanoma, NOS                          Melanoma                  1
6   ISIC_0059043        Basal cell carcinoma          Non-melanoma skin cancer                  1
7   ISIC_0063984        Basal cell carcinoma          Non-melanoma skin cancer                  1
8   ISIC_0060789       

## Block 7 — Limitations

1. **Taxonomy not yet trained** — The cancer-risk 4-class taxonomy has not yet been used to retrain the model (#119). This notebook defines requirements; actual benchmark results require trained model weights.
2. **24-image sample is qualitative only** — 24 images is not statistically significant. Full evaluation must use the frozen test split.
3. **Actinic keratosis placement** — Actinic keratosis is grouped into `Other non-cancer / indeterminate lesion`. This placement is confirmed — D3.1 finalized the 4-class taxonomy in #117.
4. **No skin-tone stratification** — BCN20000 has no Fitzpatrick skin-tone metadata. Skin-tone slice evaluation requires FitzPatrick17k or SCIN.
5. **Educational prototype — not a clinical tool** — Results must not be presented as clinical validation or diagnostic accuracy claims.
6. **Ground truth scope** — Ground truth is BCN20000 histopathology-confirmed labels. Model may not generalise to clinical photographs or other imaging modalities.

## Block 8 — Summary

| Item | Detail |
|------|--------|
| Taxonomy | 4-class cancer-risk (Option C — D3.1, finalized #117) |
| Classes | Melanoma / Non-melanoma skin cancer / Benign nevus / Other non-cancer / indeterminate lesion |
| Dataset | BCN20000 — histopathology confirmed |
| Qualitative sample | 24 images (6 per class) — `bcn20000_benchmark_24_images.csv` |
| Full eval dataset | Frozen test split — `data/processed/bcn20000_cancer_risk/test.csv` (from #118) |
| Primary safety metric | Cancer recall (Melanoma + NMSC combined) |
| Standard metrics | Accuracy, Macro-F1, Balanced Accuracy, per-class P/R/F1, Confusion Matrix |
| Output template | `bcn20000_cancer_risk_benchmark_template.csv` |

### Dependency Chain
```
D3.1 — Approve cancer-risk taxonomy  ✅ confirmed (#116, #117)
  → D3.3 (#118) — Rebuild BCN20000 splits with new taxonomy
    → D3.4 (#119) — Retrain CNN with cancer-risk taxonomy
      → D3.5 (#120) — Evaluate CNN on frozen test split  ← THIS BENCHMARK
```

### Next Steps
1. ✅ Taxonomy confirmed — D3.1 (#116) and #117 approved the 4-class taxonomy
2. Rebuild BCN20000 processed splits (#118)
3. Retrain CNN (#119)
4. Run evaluation on frozen test split using metrics in Block 5 (#120)
5. Update model evaluation report (E1.5 — #46)